# BPE

这是使用python实现BPE底层逻辑的具体实现：

In [1]:
# In[1]: 导入库与准备微型语料库
import re
from collections import defaultdict

# 为了方便演示，我们直接统计好一个微型字典库的词频
# 注意：每个单词的字符之间用空格隔开，末尾加上特殊的 </w> 符号
vocab = {
    'l o w </w>': 5,
    'l o w e s t </w>': 2,
    'n e w e r </w>': 6,
    'w i d e r </w>': 3
}

print("初始语料库状态:")
for word, freq in vocab.items():
    print(f"{word}: {freq}")

初始语料库状态:
l o w </w>: 5
l o w e s t </w>: 2
n e w e r </w>: 6
w i d e r </w>: 3


In [2]:
# In[2]: 定义 BPE 的核心统计与合并函数

def get_stats(vocab):
    """
    扫描当前语料库，统计所有相邻字符对（Pairs）的出现频率。
    """
    pairs = defaultdict(int)
    for word, freq in vocab.items():
        symbols = word.split()
        for i in range(len(symbols) - 1):
            # 记录相邻的一对符号，并累加它们的频率
            pairs[symbols[i], symbols[i+1]] += freq
    return pairs

def merge_vocab(pair, v_in):
    """
    将语料库中出现的最频繁的字符对合并为一个新的子词。
    """
    v_out = {}
    # 把 pair 组合成正则表达式模式，比如对 ('e', 'r') 组合成匹配 'e r' 的模式
    bigram = re.escape(' '.join(pair))
    # 使用正则替换，(?<!\S) 和 (?!\S) 确保我们只匹配独立的符号，不破坏已经合并的词
    p = re.compile(r'(?<!\S)' + bigram + r'(?!\S)')
    
    for word in v_in:
        # 将匹配到的 'e r' 替换为没有空格的 'er'
        w_out = p.sub(''.join(pair), word)
        v_out[w_out] = v_in[word]
    return v_out

In [3]:
# In[3]: 开启 BPE 训练循环

num_merges = 10  # 我们设置最多进行 10 次合并操作
merge_rules = [] # 用于保存合并规则，推理时需要严格按照这个顺序

print("--- 开始 BPE 合并训练 ---")
for i in range(num_merges):
    pairs = get_stats(vocab)
    
    # 如果没有可以合并的字符对（即所有的词都合并完了），就提前结束
    if not pairs:
        break
        
    # 找出频率最高的字符对
    best_pair = max(pairs, key=pairs.get)
    
    # 记录这条规则
    merge_rules.append(best_pair)
    
    # 更新语料库
    vocab = merge_vocab(best_pair, vocab)
    
    print(f"第 {i+1} 轮合并: {best_pair[0]} + {best_pair[1]} -> {''.join(best_pair)}")

print("\n--- 最终的词汇形态 ---")
for word, freq in vocab.items():
    print(f"{word}: {freq}")

--- 开始 BPE 合并训练 ---
第 1 轮合并: e + r -> er
第 2 轮合并: er + </w> -> er</w>
第 3 轮合并: l + o -> lo
第 4 轮合并: lo + w -> low
第 5 轮合并: n + e -> ne
第 6 轮合并: ne + w -> new
第 7 轮合并: new + er</w> -> newer</w>
第 8 轮合并: low + </w> -> low</w>
第 9 轮合并: w + i -> wi
第 10 轮合并: wi + d -> wid

--- 最终的词汇形态 ---
low</w>: 5
low e s t </w>: 2
newer</w>: 6
wid er</w>: 3


In [4]:
# In[4]: 编写 BPE 编码器（推理阶段）

def bpe_tokenize(text, rules):
    """
    使用训练好的规则，对未知新词进行分词。
    """
    # 1. 预处理：加 </w> 并按字符拆分加空格
    word = ' '.join(list(text)) + ' </w>'
    
    # 2. 严格按照训练时的顺序应用合并规则
    for pair in rules:
        bigram = re.escape(' '.join(pair))
        p = re.compile(r'(?<!\S)' + bigram + r'(?!\S)')
        word = p.sub(''.join(pair), word)
        
    # 去掉中间的空格，将拆分好的子词放入列表
    return word.split()

# 测试一个训练集里没见过的新词
new_word = "lower"
tokens = bpe_tokenize(new_word, merge_rules)
print(f"新词 '{new_word}' 被分词为: {tokens}")

新词 'lower' 被分词为: ['low', 'er</w>']


# 简洁实现

使用Hagging Face的tokenizers实现。 

In [ ]:
!pip install tokenizers --quiet

In [6]:
# In[5]: 使用工业级库极其优雅地实现 BPE

from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace

# 1. 初始化一个基于 BPE 模型的空分词器
# unk_token 用于兜底那些连基础字符都没见过的极端情况
tokenizer = Tokenizer(BPE(unk_token="[UNK]"))

# 2. 设置预分词器 (告诉模型先用空格把句子切成单词，再进行 BPE 切分)
tokenizer.pre_tokenizer = Whitespace()

# 3. 初始化 BPE 训练器
# 我们可以设定期望的最终词表大小，以及一些特殊 Token
trainer = BpeTrainer(
    vocab_size=100, # 演示用，真实场景通常是 30000 - 50000
    special_tokens=["[UNK]", "[PAD]", "[CLS]", "[SEP]", "[MASK]"]
)

# 4. 准备一个迷你的文本文件来模拟海量语料
with open("bpe_dummy_corpus.txt", "w", encoding="utf-8") as f:
    f.write("low low low low low\n")
    f.write("lowest lowest\n")
    f.write("newer newer newer newer newer newer\n")
    f.write("wider wider wider\n")

# 5. 一键起飞：直接在文件上训练
print("开始训练工业级 BPE 分词器...")
tokenizer.train(files=["bpe_dummy_corpus.txt"], trainer=trainer)

# 6. 测试编码
test_text = "lower and wider"
encoded = tokenizer.encode(test_text)

print(f"\n原始句子: '{test_text}'")
print(f"切分后的子词 (Tokens): {encoded.tokens}")
print(f"对应的 PyTorch 可用的 ID: {encoded.ids}")

# 保存下来给后续的 PyTorch 模型加载使用
# tokenizer.save("my_bpe_tokenizer.json")

开始训练工业级 BPE 分词器...

原始句子: 'lower and wider'
切分后的子词 (Tokens): ['low', 'er', '[UNK]', 'n', 'd', 'wider']
对应的 PyTorch 可用的 ID: [17, 15, 0, 9, 5, 23]
